# Buyer Dataset Preparation

This notebook creates and validates a dedicated **confirmed buyers** dataset from the cleaned Deals table.

The buyer subset is used for analyses where payment and study fields are meaningful, including:
- product performance
- payment behavior
- revenue and unit economics
- education type
- student-level geography

The goal is to preserve uncertain CRM records while correcting only anomalies that can be resolved with strong evidence.

> **Data note:** the original CRM files are not included in the public repository.


In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

pd.options.display.max_columns = None


The cleaned Deals table contains all potential deals, while only a relatively small subset has sufficient evidence of both payment and active study.

Fields such as `product`, `education_type`, `payment_type`, `months_of_study`, `course_duration`, `initial_amount_paid`, and `offer_total_amount` are primarily meaningful for actual students.

For buyer-specific financial and product analysis, this notebook therefore creates a separate dataset containing only records with `is_buyer = True`.

In [ ]:
deals = pd.read_pickle(PROCESSED_DIR / 'deals_clean.pkl')
contacts_id_replacements = pd.read_pickle(
    PROCESSED_DIR / 'contacts_id_replacements.pkl'
)

In [ ]:
buyers = deals[deals['is_buyer']].copy()

In [ ]:
buyers.isna().sum()

## 1. Logical Consistency Checks

### 1.1. `initial_amount_paid` > `offer_total_amount`

In [ ]:
anomaly_init_offer = buyers['initial_amount_paid'] > buyers['offer_total_amount']
print(anomaly_init_offer.sum())

In [ ]:
buyers[anomaly_init_offer]

Only one clearly identifiable amount-entry error is corrected: `initial_amount_paid = 11000` and `offer_total_amount = 1200` are swapped.

Other cases where the initial amount exceeds the offer amount are retained because the available data does not provide enough evidence to determine the correct value reliably.

In [ ]:
# Correct only the clearly identifiable swapped amount pair
swap_mask = (
    (buyers['initial_amount_paid'] == 11000)
    & (buyers['offer_total_amount'] == 1200)
)

print(f'Rows to correct: {swap_mask.sum()}')

buyers.loc[swap_mask, 'initial_amount_paid'] = 1200
buyers.loc[swap_mask, 'offer_total_amount'] = 11000

buyers.loc[
    swap_mask,
    ['id', 'initial_amount_paid', 'offer_total_amount']
]

### 1.2 `closing_date` < `created_time`

In [ ]:
anomaly_date = buyers['closing_date'].dt.date < buyers['created_time'].dt.date
print(anomaly_date.sum())

In [ ]:
buyers[anomaly_date]

The negative deal duration had already been prevented in the Deals cleaning step by leaving `deal_duration_days` missing for invalid date sequences.

### 1.3. `lost_reason`

In [ ]:
buyers['lost_reason'].value_counts()

In [ ]:
buyers[buyers['lost_reason'].notna()][['id', 'lost_reason', 'stage', 'initial_amount_paid', 'deal_duration_days']]

Some confirmed buyers still contain a non-empty `lost_reason`. Because their deal stage is `Payment Done`, these values are treated as historical CRM artifacts rather than evidence that the buyer should be removed.

When analyzing lost reasons, confirmed buyers should be excluded from that analysis.

### 1.4. `months_of_study` > `course_duration` 

In [ ]:
anomaly_months = buyers['months_of_study'].astype('Int64') > buyers['course_duration'].astype('Int64')
print(f'months_of_study > course_duration: {anomaly_months.sum()}')
buyers[anomaly_months][['contact_name', 'product', 'course_duration', 'months_of_study']]

All confirmed buyers remain within the recorded course duration (`months_of_study <= course_duration`).

## 2. Repeated Buyer Contacts

In [ ]:
buyers[buyers.duplicated(subset='contact_name', keep=False)].sort_values('contact_name')

A small number of contacts appear more than once in the buyer dataset.

Some repeat records represent purchases of different products, which may be valid repeat purchases. Other repeated product records cannot be resolved confidently without external business confirmation.

Because these cases represent only a very small share of the buyer dataset and may contain real sales, all records are retained.

In [ ]:
# Verify whether any buyer still uses an old contact ID
buyers_in_replacements = buyers['contact_name'].isin(
    contacts_id_replacements.keys()
)

print(
    'Buyer contacts requiring ID replacement:',
    buyers_in_replacements.sum()
)

buyers.loc[
    buyers_in_replacements,
    ['id', 'contact_name']
]

All buyer contact IDs are already retained / main contact IDs according to the contact deduplication mapping.

## 3. Missing Values

In [ ]:
print(buyers[buyers['payment_type'].isna()]['product'].value_counts())

`payment_type` contains substantial missingness among confirmed buyers.

The missing values are distributed across products and cannot be reconstructed reliably from the available CRM fields. They are therefore retained as `NaN`.

Payment-type analysis should use records where `payment_type` is available and state the resulting denominator explicitly.

In [ ]:
buyers[buyers['education_type'].isna()][['product', 'course_duration', 'months_of_study', 'closing_date']]

In [ ]:
buyers_marketing = buyers[buyers['product'] == 'Digital Marketing']
buyers_marketing.groupby('education_type')['course_duration'].value_counts().sort_index()

A small number of `education_type` values are missing, all in early Digital Marketing records.

Because there is no reliable basis for reconstruction, the values are retained as missing and are naturally excluded from education-type group comparisons.

## 4. Save Buyer Dataset

In [ ]:
# Pickle preserves parsed data types for downstream notebooks
buyers.to_pickle(PROCESSED_DIR / 'buyers.pkl')

print('Saved: buyers.pkl')
print(f'Shape: {buyers.shape}')